In [ ]:
!python3 -m pip install scikit-learn pandas numpy matplotlib xgboost

In [ ]:
from xgboost import XGBClassifier
import xgboost
print(xgboost.__version__)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier


# Data Explore
## Goal：to predict 三种 cancer 「GBM, LUSC, and OV.」
- Data are in log space.
- n=886 samples: GBM:376, LUSC: 90, OV= 420 samples.

Note: For this project, we manually **included a small amount of data issues** to better reflect real-world settings. 

// For example, we added some errors and flipped a subset of labels in the training data. You may consider strategies to identify and handle potentially mislabeled observations in your final project. If you do so, please clearly describe what you did and why.

In [ ]:
#import pandas as pd
train = pd.read_csv('train.csv')

In [ ]:
n, p = train.shape
print(f"nobsv = {n}, predictors = {p}")

In [ ]:
#train.describe(include='all')

In [ ]:
train.head()

### - Null

In [ ]:
print(f"null value = {sum(train.isnull().sum()) }")

### - Predictors type

In [ ]:
train.dtypes

In [ ]:
a = []
for col in train.columns:
    if train[col].dtype != 'float64':
        a.append(col)
print(f"category = {a}")

### - Response: Y

In [ ]:
train.cancer.describe() # continuos才管用
train.cancer.value_counts() 
# 原本：GBM:376, LUSC: 90, OV= 420 samples.
# total_n=886,GBM=0.4244, LUSC=0.1016, OV=0.4740
print(f"""total_n = {sum(train.cancer.value_counts())},
class ratio = {train.cancer.value_counts(1)}, 
class counts = {train.cancer.value_counts()}
""")

### Features: X

In [ ]:
temp = train.copy()
xs = temp.drop(columns=['cancer'])
xs.describe()

In [ ]:
train = pd.read_csv('train.csv')
x_all = train.drop(columns=['cancer', 'id'])
y_all = train['cancer']
y_all = pd.DataFrame(
    y_all,
    index=y_all.index)

df_plot = x_all.copy()
df_plot["class"] = y_all["cancer"]

import matplotlib.pyplot as plt
import seaborn as sns

# TP63
plt.figure()
sns.boxplot(x="class", y="TP63", data=df_plot)
plt.title("TP63 by class")
plt.show()

# C1QL1
plt.figure()
sns.boxplot(x="class", y="C1QL1", data=df_plot)
plt.title("C1QL1 by class")
plt.show()

## Data Prepare
### - Random Split with Stratified

In [ ]:
#from sklearn.model_selection import train_test_split
train_d, test_d = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train['cancer'])
# 检查一下ratio
train_d.head()
test_d.head()

print(f"""train counts = {sum(train_d.cancer.value_counts())}, 
train ratio = {train_d.cancer.value_counts(1)},
class counts = {train_d.cancer.value_counts()},
test counts = {sum(test_d.cancer.value_counts())}, 
test ratio = {test_d.cancer.value_counts(1)},
class counts = {test_d.cancer.value_counts()}
""")

### - Standardization: X
 $p = \frac{p - \hat{p}}{std}$.

In [ ]:
x_train = train_d.drop(columns=['cancer', 'id'])
y_train = train_d['cancer']
y_train = pd.DataFrame(
    y_train,
    index=y_train.index)

x_test = test_d.drop(columns=['cancer', 'id'])
y_test = test_d['cancer']
y_test = pd.DataFrame(
    y_test,
    index=y_test.index)

In [ ]:
x_train.head()

In [ ]:
y_train.head()

In [ ]:
#from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(x_train)

x_train_mid = scaler.transform(x_train)
x_test_mid = scaler.transform(x_test)
# 这俩出来是numpy.ndarray

In [ ]:
# 所以要给他搞成 dataframe
#import pandas as pd
x_train_scaled = pd.DataFrame(
    x_train_mid,
    columns=x_train.columns,
    index=x_train.index)

x_test_scaled = pd.DataFrame(
    x_test_mid, 
    columns=x_test.columns,
    index=x_test.index)

#x_train_scaled.head()

In [ ]:
#x_train_scaled.describe()
print(f"""
train mean = {abs(x_train_scaled.mean()).max()},
train std = {abs(x_train_scaled.std()).max()},
mean and std has been standardized
""")
print(f"""
test mean = {abs(x_test_scaled.mean()).max()},
test std = {abs(x_test_scaled.std()).max()},
mean and std has been FIT according to train's mean and std
""")

In [ ]:
plt.figure(figsize=(10,4))

mean_before = np.mean(x_train['RNF14'])
var_before = np.var(x_train['RNF14'])
mean_after = np.mean(x_train_scaled['RNF14'])
var_after = np.var(x_train_scaled['RNF14'])

# Before
plt.subplot(1,2,1)
plt.hist(x_train['RNF14'], bins=30,color = 'skyblue')
plt.axvline(mean_before, linestyle='--')
plt.text(mean_before, plt.ylim()[1]*0.9,
         f"Mean={mean_before:.2f}\nVar={var_before:.2f}")
plt.title('Before Standardization')
plt.xlabel('RNF14')

# After
plt.subplot(1,2,2)
plt.hist(x_train_scaled['RNF14'], bins=30,color = 'skyblue')

plt.axvline(mean_after, linestyle='--')
plt.text(mean_after, plt.ylim()[1]*0.9,
         f"Mean={mean_after:.2f}\nVar={var_after:.2f}")

plt.title('After Standardization')
plt.xlabel('RNF14 (standardized)')

plt.tight_layout()
plt.show()


### - Response Visualization: Y

In [ ]:
y_train.value_counts().plot(kind='bar')

plt.xlabel("Cancer class")
plt.ylabel("Counts")
plt.title("trian Class distribution")
plt.show()

## 问题
1. 目前我们的data 的predictros X 比 observation 多，p>n
2. 有一小部分样本的Cancer lable是错的，Predictors是对的，但 class label 是错误的。这就是GBM:376, LUSC: 90, OV= 420 samples和我们真是的train data cancer counts不一样

以下是我们train要使用的 y 和 x ：

In [ ]:
type(y_train['cancer'])

In [ ]:
x_train_scaled.head()

In [ ]:
print(f"""trian(n,p)={x_train_scaled.shape}
test(n,p)={x_test_scaled.shape}""")

# Model: 别跑这一段，直接去跑下一段
### Preliminary Model: M0- Lasso Selection


In [ ]:
# C = 1/lambda；C 越小 => penalty则越强
Cs_grid = np.logspace(-3, 2, 10)

lasso_cv = LogisticRegressionCV(
    Cs=Cs_grid,
    penalty='l1',
    solver='saga',        # L1 必须用 saga
    cv=3,                 # 先用 3-fold 更快；稳定后再换 5
    scoring='accuracy',
    max_iter=8000,
    n_jobs=-1,
    refit=True)


In [ ]:
lasso_cv.fit(x_train_scaled, y_train['cancer']) 

In [ ]:
# Selected feature
selected_features = x_train_scaled.columns[(lasso_cv.coef_ != 0).any(axis=0)]

# Print
print("Selected features:")
print(selected_features.tolist())

# Export to .csv
pd.Series(selected_features).to_csv("lasso_selected_features.csv", index=False)

In [ ]:
for i, cls in enumerate(lasso_cv.classes_):
    print(f"class {cls} nonzero coef:{np.sum(lasso_cv.coef_[i] != 0)}")


In [ ]:
print("Selected C (1/lambda) for each class:")
for cls, c in zip(lasso_cv.classes_, lasso_cv.C_):
    print(f"class {cls}: C = {c}")

print("\nSelected regularization values:")
for cls, c in zip(lasso_cv.classes_, lasso_cv.C_):
    print(f"class {cls}: C = {c:.5f}, lambda = {1/c:.5f}")

print("\nFinal model C values:", lasso_cv.C_)

print("\nGrid of C values searched:")
print(lasso_cv.Cs_)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

coef_matrix = pd.DataFrame(
    lasso_cv.coef_,
    columns=x_train_scaled.columns,
    index=[f"Class {c}" for c in lasso_cv.classes_]
)

#不是0的coef留着
selected = (coef_matrix != 0).any(axis=0)
coef_matrix = coef_matrix.loc[:, selected]
importance = coef_matrix.abs().max(axis=0)
top_features = importance.sort_values(ascending=False).head(20).index#再找排名高的
coef_matrix = coef_matrix[top_features]

#图
plt.figure(figsize=(12,6))
plt.imshow(coef_matrix, aspect='auto')
plt.colorbar(label='Coefficient')

plt.xticks(range(len(coef_matrix.columns)), coef_matrix.columns, rotation=90)
plt.yticks(range(len(coef_matrix.index)), coef_matrix.index)

plt.title("Lasso Coefficient Heatmap (Selected Features)")
plt.tight_layout()
plt.show()

# Formal Model：从这里开跑
### Train/Test Data with Selected Features:

In [ ]:
selected_features = pd.read_csv('lasso_selected_features.csv')
selected_features['0']

以下是 Standardize + Lasso Selection + train/test：

In [ ]:
x_train_selected = x_train_scaled[selected_features['0']]
x_test_selected  = x_test_scaled[selected_features['0']]
print(f"""X: x_train_selected  x_test_selected 
Y: y_train           y_test""")

In [ ]:
print(f"""trian(n,p)={x_train_selected.shape}
test(n,p)={x_test_selected.shape}
after selection, predictors reduced to total 65 and observations stay same """)

### Full Data with selected Features:

In [ ]:
train = pd.read_csv('train.csv')
x_all = train.drop(columns=['cancer', 'id'])
y_all = train['cancer']
y_all = pd.DataFrame(
    y_all,
    index=y_all.index)

In [ ]:
# 这里在standardization
scaler = StandardScaler()
scaler.fit(x_all)
x_all_mid = scaler.transform(x_all)
# 这俩出来是numpy.ndarray

# 所以要给他搞成 dataframe
#import pandas as pd
x_all_scaled = pd.DataFrame(
    x_all_mid,
    columns=x_all.columns,
    index=x_all.index)
#x_all_scaled.head()

以下是 Standardize + Lasso Selection + ALL data：

In [ ]:
x_all_selected = x_all_scaled[selected_features['0']]

print(f"""X: x_all_selected
Y: y_all""")

# M1: Ridge

In [ ]:
m1 = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=5000)

m1.fit(x_train_selected, y_train['cancer'])

In [ ]:
y_pred_train = m1.predict(x_train_selected)
print(f"""train counts:\n{pd.Series(y_pred_train).value_counts()},
test counts:{y_train['cancer'].value_counts()}""")

In [ ]:
acc_train = accuracy_score(y_train['cancer'], y_pred_train)
print("Train Accuracy:", acc_train)

y_pred_test = m1.predict(x_test_selected)
acc_test = accuracy_score(y_test['cancer'], y_pred_test)
print("Test Accuracy:", acc_test)

In [ ]:
# from sklearn.model_selection import cross_val_predict

proba_cv = cross_val_predict(
    m1,
    x_train_selected,
    y_train['cancer'],
    cv=5,
    method="predict_proba")

#_______________________________________

classes = m1.classes_ # array([1, 2, 3])
pred_cv = classes[np.argmax(proba_cv, axis=1)]

confidence = proba_cv.max(axis=1)


#_______________________________________

check = pd.DataFrame({
    "true_label": y_train['cancer'],
    "pred": pred_cv,
    "confidence": confidence })

#_______________________________________

wrong = check[check.true_label != check.pred]
suspect = wrong.sort_values("confidence", ascending=False)
suspect = wrong[wrong.confidence > 0.9]
suspect.shape # (34, 3)

In [ ]:
suspect.index

# M2: 改全部的mislabeled

In [ ]:
# 把东西给flip回来
y_train_clean = y_train.copy()
idx = suspect.index
y_train_clean.loc[idx, 'cancer'] = suspect.loc[idx, 'pred']
idx = suspect.index
((y_train_clean.loc[idx, "cancer"] != y_train.loc[idx, "cancer"]).sum())

In [ ]:
m2 = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=5000)

m2.fit(x_train_selected, y_train_clean['cancer'])

y_pred_train2 = m2.predict(x_train_selected)

In [ ]:
print(f"""train counts:\n{pd.Series(y_pred_train2).value_counts()},
test counts:{y_train_clean['cancer'].value_counts()}""")


In [ ]:
acc_train = accuracy_score(y_train_clean['cancer'], y_pred_train2)
print("Train Accuracy:", acc_train)

y_pred_test2 = m2.predict(x_test_selected)
acc_test = accuracy_score(y_test['cancer'], y_pred_test2)
print("Test Accuracy:", acc_test)

# M3+M4: 不用train test，全部来train

In [ ]:
m3 = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=5000)

m3.fit(x_all_selected, y_all['cancer'])

y_pred_train3 = m3.predict(x_all_selected)

In [ ]:
# from sklearn.model_selection import cross_val_predict

proba_cv = cross_val_predict(
    m3,
    x_all_selected,
    y_all['cancer'],
    cv=5,
    method="predict_proba")

#_______________________________________

classes = m3.classes_ # array([1, 2, 3])
pred_cv = classes[np.argmax(proba_cv, axis=1)]

confidence = proba_cv.max(axis=1)


#_______________________________________

check = pd.DataFrame({
    "true_label": y_all['cancer'],
    "pred": pred_cv,
    "confidence": confidence })

#_______________________________________

wrong = check[check.true_label != check.pred]
suspect = wrong.sort_values("confidence", ascending=False)
suspect = wrong[wrong.confidence > 0.9]
suspect.head()#(47, 3)

In [ ]:
# 把东西给flip回来
y_all_clean = y_all.copy()
idx = suspect.index
y_all_clean.loc[idx, 'cancer'] = suspect.loc[idx, 'pred']
((y_all_clean.loc[idx, "cancer"] != y_all.loc[idx, "cancer"]).sum())

In [ ]:
m4 = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    max_iter=5000)

m4.fit(x_all_selected, y_all_clean ['cancer'])
y_pred_all4 = m4.predict(x_all_selected)


In [ ]:
#from sklearn.metrics import ConfusionMatrixDisplay
#import matplotlib.pyplot as plt

acc_train_4 = accuracy_score(y_all_clean ['cancer'], y_pred_all4 )
print("Train all Accuracy:", acc_train_4 )

ConfusionMatrixDisplay.from_predictions(y_all_clean['cancer'],y_pred_all4)
plt.title("Confusion Matrix")
plt.show()

# Define a function to calculate the type-I and type-II error rate 
Since this is a 3-class classification problem, typical definition of type-I and type-II error rate does not apply here.

We instead adopt a one-vs-rest formulation. For each class, we treat it as a binary classification problem:

|True/Pred| k | Not k|
|---------|---|------|
| k | TP | FN |
| Not k | FP | TN |

Under this definition:
 - the Type-I error rate for class-k is: 
 $P(\hat{Y} = k | Y \neq k)$
 - the Type-II error rate for class-k is: 
 $P(\hat{Y} \neq k | Y = k)$

We define the mean Type-I/Type-II error rates as the average of the class-wise error rates across all classes:

$$
\text{Mean Type-I Error}
=
\frac{1}{K}
\sum_{k=1}^{K}
P(\hat{Y} = k \mid Y \neq k)
$$

$$
\text{Mean Type-II Error}
=
\frac{1}{K}
\sum_{k=1}^{K}
P(\hat{Y} \neq k \mid Y = k)
$$

*Note that: this definition applies not only to 3-class classification, but also apply to any number of class more than 3.*


In [ ]:
# Function to calculate type-I and type-II error for n class classification problem.
def CalculateTypeError(real_val, pred_val) -> str:
    real_val = np.asarray(real_val)
    pred_val = np.asarray(pred_val)
    total = len(real_val)    
    if(len(real_val) != len(pred_val)):
        return "Cannot calculate type error: length of real values and predicted values mismatch."
    
    classes_list = list(np.unique(real_val))
    classes_list.sort()
    num_classes = len(classes_list)

    type_I_cnt = [0] * num_classes
    type_II_cnt = [0] * num_classes

    for i in range(total):
        real = real_val[i]
        pred = pred_val[i]
        index_II = classes_list.index(real)
        index_I = classes_list.index(pred)
        if (real != pred):
            type_I_cnt[index_I] += 1
            type_II_cnt[index_II] += 1
    
    classes_cnt = [0] * num_classes
    for k in real_val:
        index_II = classes_list.index(k)
        classes_cnt[index_II] += 1
    
    print("\nType-I/Type-II Error Rate:\n")
    for i in range(num_classes):
        print(f"class {classes_list[i]}:")
        print(f"Type-I error rate: {type_I_cnt[i] / (total - classes_cnt[i])} \nType-II error rate: {type_II_cnt[i] / classes_cnt[i]}")
    total_type_I_cnt = 0
    total_type_II_cnt = 0

    # Mean rate
    type_I_rates = []
    type_II_rates = []

    for i in range(num_classes):
        type_I = type_I_cnt[i] / (total - classes_cnt[i])
        type_II = type_II_cnt[i] / classes_cnt[i]
        type_I_rates.append(type_I)
        type_II_rates.append(type_II)

    print(f"\nMean Type-I error rate: {np.mean(type_I_rates)}")
    print(f"Mean Type-II error rate: {np.mean(type_II_rates)}")

# M5: trian/test LDA QDA

In [ ]:
# Checking Gaussian assumption
import scipy.stats as stats
import matplotlib.pyplot as plt

feature = x_train.columns[0]

stats.probplot(x_train[feature], dist="norm", plot=plt)
plt.title(f"Q-Q plot for {feature}")
plt.show()

In [ ]:
# LDA/QDA
# import libraries
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Model fitting (LDA)
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(x_train_selected, y_train["cancer"])
y_pred_train_lda = lda_model.predict(x_train_selected)

# Model fitting (QDA)
qda_model = QuadraticDiscriminantAnalysis()
qda_model.fit(x_train_selected, y_train["cancer"])
y_pred_train_qda = qda_model.predict(x_train_selected)


In [ ]:
# Correct mislabled data using M5
proba_cv = cross_val_predict(
    lda_model,
    x_train_selected,
    y_train['cancer'],
    cv=5,
    method="predict_proba")

#_______________________________________

classes = lda_model.classes_ # array([1, 2, 3])
pred_cv = classes[np.argmax(proba_cv, axis=1)]

confidence = proba_cv.max(axis=1)


#_______________________________________

check = pd.DataFrame({
    "true_label": y_train['cancer'],
    "pred": pred_cv,
    "confidence": confidence })

#_______________________________________

wrong = check[check.true_label != check.pred]
suspect = wrong.sort_values("confidence", ascending=False)
suspect = wrong[wrong.confidence > 0.9]
suspect.head()#(47, 3)

In [ ]:
# 把东西给flip回来
y_train_clean = y_train.copy()
idx = suspect.index
y_train_clean.loc[idx, 'cancer'] = suspect.loc[idx, 'pred']
((y_train_clean.loc[idx, "cancer"] != y_train.loc[idx, "cancer"]).sum())

In [ ]:
# 重新fit LDA
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(x_train_selected, y_train_clean["cancer"])
y_pred_train_lda = lda_model.predict(x_train_selected)
type(y_pred_train_lda)
type(y_train["cancer"])

### Accuracy: Trian

In [ ]:
acc_train_lda = accuracy_score(y_train["cancer"], y_pred_train_lda)
print("Train LDA Accuracy:", acc_train_lda)

ConfusionMatrixDisplay.from_predictions(y_train["cancer"],y_pred_train_qda)
plt.title("Trian LDA-Confusion Matrix")
plt.show()

CalculateTypeError(y_train["cancer"], y_pred_train_lda)

In [ ]:
acc_train_qda = accuracy_score(y_train["cancer"], y_pred_train_qda )
print("Train QDA Accuracy:", acc_train_qda)

ConfusionMatrixDisplay.from_predictions(y_train["cancer"],y_pred_train_qda)
plt.title("Train QDA-Confusion Matrix")
plt.show()

### Accuracy: Test

In [ ]:
y_pred_test_lda = lda_model.predict(x_test_selected)
acc_test_lda = accuracy_score(y_test["cancer"], y_pred_test_lda )
print("Test LDA Accuracy:", acc_test_lda)

ConfusionMatrixDisplay.from_predictions(y_test["cancer"],y_pred_test_lda)
plt.title("Test LDA-Confusion Matrix")
plt.show()

In [ ]:
y_pred_test_qda = qda_model.predict(x_test_selected)
acc_test_qda = accuracy_score(y_test["cancer"], y_pred_test_qda )
print("Test QDA Accuracy:", acc_test_qda)

ConfusionMatrixDisplay.from_predictions(y_test["cancer"],y_pred_test_qda)
plt.title("Test QDA-Confusion Matrix")
plt.show()

# Actual Test----------------------

## M4 on test data

In [ ]:
actual_test =  pd.read_csv('test.csv')
actual_test=actual_test.drop(columns=['id'])
actual_test.head()
actual_test_mid = scaler.transform(actual_test)

actual_test_scaled = pd.DataFrame(
    actual_test_mid, 
    columns=actual_test.columns,
    index=actual_test.index)
actual_test.shape

In [ ]:
actual_test_scaled.head()

In [ ]:
actual_test_selected  = actual_test_scaled[selected_features['0']]
actual_test_selected.head()

In [ ]:
y_actual_test = m4.predict(actual_test_selected)
pd.Series(y_actual_test).value_counts()

In [ ]:

test = pd.read_csv("test.csv")
id_col = test["id"]

submission = pd.DataFrame({
    "id": id_col,
    "cancer": y_actual_test})

submission.shape

In [ ]:

submission.to_csv("submission_LDA2.csv", index=False)

## M5 on test data

In [ ]:
# predict the test data
test_predict = lda_model.predict(actual_test_selected)
pd.Series(test_predict).value_counts()

## M7:XGBoost

LASSO Feature Selection  
        ↓
XGBoost Model Training  
        ↓
5-Fold Cross Validation  
        ↓
Detect Mislabeled Samples  
        ↓
Remove Noisy Observations  
        ↓
Retrain Model  
        ↓
Test Set Prediction  
        ↓
Literature Validation (TCGA)  
        ↓
Final Submission


In [ ]:
# ---------------------------
# XGBoost + 5-fold CV tuning
# ---------------------------
#!pip install xgboost
#from xgboost import XGBClassifier
#from sklearn.model_selection import StratifiedKFold, GridSearchCV

xgb_base = XGBClassifier(        #基础 XGBoost 分类器
    objective='multi:softprob',  #多分类问题>3
    num_class=3,
    eval_metric='mlogloss',      #multi-class log loss 来评估, loss越小越好
    random_state=42,             #random seed
    n_jobs=-1,                   #训练会更快
    tree_method='hist'           #XGBoost 的一种 加速算法，速度快，省内存
)

param_grid = {
    'n_estimators': np.arange(100, 301, 50),     
    'max_depth': np.arange(3, 8),                
    'learning_rate': np.linspace(0.01, 0.1, 5),  
    'subsample': np.linspace(0.7, 1.0, 4),       
    'colsample_bytree': np.linspace(0.7, 1.0, 4)
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(  #寻找最好的x组合
    estimator=xgb_base, #xgboost
    param_grid=param_grid,
    scoring='accuracy',  #寻找最大
    cv=cv5,
    n_jobs=-1,
    verbose=1
)


#变成1D array->xgboost要求
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.iloc[:, 0]

y_encoded = y_train.astype(int) - 1  #XGBoost 多分类要求标签从 0 开始
print(np.unique(y_encoded))



###不要跑

In [ ]:
#不要跑，要30分钟
#grid.fit(x_train_selected, y_encoded) #开始寻找

#best_xgb_long = grid.best_estimator_ #提取

#print("Best params:", grid.best_params_)
#print("Best 5-fold CV accuracy:", grid.best_score_)


In [ ]:
#不要跑
#print(best_xgb_long)
#output：
# XGBClassifier(base_score=None, booster=None, callbacks=None,
#               colsample_bylevel=None, colsample_bynode=None,
#               colsample_bytree=np.float64(0.9), device=None,
#               early_stopping_rounds=None, enable_categorical=False,
#               eval_metric='mlogloss', feature_types=None, feature_weights=None,
#               gamma=None, grow_policy=None, importance_type=None,
#               interaction_constraints=None, learning_rate=np.float64(0.1),
#               max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
#               max_delta_step=None, max_depth=np.int64(4), max_leaves=None,
#               min_child_weight=None, missing=nan, monotone_constraints=None,
#               multi_strategy=None, n_estimators=np.int64(250), n_jobs=-1,
#               num_class=3, ...)


###从这开始跑

In [ ]:
#直接用上一步得到最优组合创建
best_xgb = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.1,
    colsample_bytree=0.9,
    subsample=0.9,
    random_state=42
)

best_xgb.fit(x_train_selected, y_encoded)

In [ ]:
#更efficiency的方法
#用 early stopping 自动找到“最合适的树的数量（n_estimators），避免过拟合 + 加速训练

#数据拆分
X_tr, X_val, y_tr, y_val = train_test_split(
    x_train_selected, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)
#模型设置，最多可以长到1000棵，但不一定全部用
#如果验证集 performance 连续30轮没有提升 → 停止训练
xgb_es = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=30
)

#fit
xgb_es.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(xgb_es.best_iteration)
#在第 87 棵树的时候，模型在 validation set 上表现最好


In [ ]:
best_xgb_new = XGBClassifier(
    n_estimators=87,
    max_depth=3,
    learning_rate=0.05,
    colsample_bytree=0.8,
    subsample=0.9,
    random_state=42
)

best_xgb_new.fit(x_train_selected, y_encoded)

# Cross Validation for Model Selection

In [ ]:
# 寻找 mislabeled data
from sklearn.model_selection import cross_val_predict


cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OOF predicted probabilities 用交叉验证为每一个样本生成预测结果->每个样本都是被“没见过它的模型”预测的。
# Cross-Validation OOF Prediction 每个样本的预测
# 必须来自
# 没见过这个样本的模型
#避免 overfitting，得到更真实的预测
oof_proba = cross_val_predict(
    best_xgb,
    x_train_selected,
    y_encoded,
    cv=cv5,
    method='predict_proba', #返回的是三种癌症的概率
    n_jobs=-1
)

classes = np.array([0, 1, 2])

# Argmax classification-Bayes optimal decision rule
# OOF predicted class：把预测概率转成最终预测类别，原来是（0.2，0.7，0.1）->1
oof_pred = classes[np.argmax(oof_proba, axis=1)]

# Confident learning，模型对“预测类别”的置信度 ->选择可能性最大的癌症类型
confidence = oof_proba.max(axis=1)

# 整理结果
check_df = pd.DataFrame({
    'sample_id': x_train_selected.index,
    'true_label_012': y_encoded,
    'true_label_123': y_encoded + 1,
    'pred_label_012': oof_pred,
    'pred_label_123': oof_pred + 1,
    'confidence': confidence
}, index=x_train_selected.index)

# 找预测错误的样本
wrong_df = check_df[check_df['true_label_012'] != check_df['pred_label_012']]

# 模型非常确定（大于90%概率确定）+ 标签不同->原来的标签可能是错的
confidence_threshold = 0.90
suspicious_df = wrong_df[wrong_df['confidence'] > confidence_threshold].sort_values(
    by='confidence',
    ascending=False
)

print("Number of suspicious samples:", suspicious_df.shape[0])
display(suspicious_df.head(10))



In [ ]:
#56个大概错误的数据
#56 / 886 ≈ 6% 可直接删除
len(suspicious_df)

Remove mislabled data

In [ ]:
X_train_original = x_train_selected.copy()
y_train_original = y_encoded.copy()

In [ ]:
remove_idx = suspicious_df.index

X_clean = X_train_original.drop(index=remove_idx)
y_clean = y_train_original.drop(index=remove_idx)

In [ ]:
print("Original samples:", X_train_original.shape[0])
print("Removed samples:", len(remove_idx))
print("Remaining samples:", X_clean.shape[0])

Retrain using the dataset that removed mislabled data

In [ ]:
#用 clean dataset 重新训练模型
best_xgb.fit(X_clean, y_clean)

#Accuracy score for old model(before drop mislabled data) and new model(train without mislabled data)

In [ ]:
from sklearn.model_selection import cross_val_score
y_all_encoded =  y_all - 1
scores_old = cross_val_score(
    best_xgb,
    x_train_selected,
    y_encoded,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)
scores_new = cross_val_score(
    best_xgb,
    X_clean,
    y_clean,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)
scores_old_on_full = cross_val_score(
    best_xgb,
    x_all_selected,
    y_all_encoded,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)
scores_new_new =cross_val_score(
    best_xgb_new,
    x_train_selected,
    y_encoded,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)
scores_new_new_on_drop =cross_val_score(
    best_xgb_new,
    X_clean,
    y_clean,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)
scores_new_new_on_full =cross_val_score(
    best_xgb_new,
    x_all_selected,
    y_all_encoded,
    cv=cv5,
    scoring="accuracy",
    n_jobs=-1
)

print("m1-train-on-drop-mis CV accuracy:", scores_old.mean())
print("m1-train-on-drop-mis CV accuracy on clean dataset:", scores_new.mean())
print("m1-train-on-drop-mis CV accuracy on all dataset:", scores_old_on_full.mean())
print("m2 CV accuracy:", scores_new_new.mean())
print("m2 CV accuracy on clean dataset:", scores_new_new_on_drop.mean())
print("m2 CV accuracy on all dataset:", scores_new_new_on_full.mean())
#Old CV accuracy: 0.9025871541304566
#New CV accuracy: 0.9831004110393422
#New-New CV accuracy: 0.9054040555389072
#accuracy imporve a lot after dropping mislabled data

In [ ]:
#test
test = pd.read_csv("test.csv")
id_col = test["id"]
test_x = test.drop(columns=["id"]) #无id

test_x_scaled = pd.DataFrame(
    scaler.transform(test_x),
    columns=test_x.columns,
    index=test_x.index
) #scale test里面的x

test_x_selected = test_x_scaled[selected_features['0']]

test_pred_xgb = best_xgb_new.predict(test_x_selected)
test_pred_xgb_old = best_xgb.predict(test_x_selected)

Submission .csv for XGBoost

In [ ]:
# 如果训练时用了 0,1,2 编码，需要 +1
test_pred_123 = test_pred_xgb + 1
test_pred_123_old = test_pred_xgb_old + 1

# 生成 submission
submission_xgb = pd.DataFrame({
    "id": id_col,
    "label": test_pred_123
})
submission_xgb_old = pd.DataFrame({
    "id": id_col,
    "label": test_pred_123_old
})
# 导出 csv
submission_xgb.to_csv("submission_xgb_new.csv", index=False)
submission_xgb_old.to_csv("submission_xgb_old.csv", index=False)
submission_xgb.to_csv("submission_xgb_new1.csv", index=False)

## M8：Random Forest 

In [ ]:
# 1: Prepare data for RF tuning

X_rf = x_train_selected.copy()
y_rf = y_train.copy()

print("X shape:", X_rf.shape)
print("Class counts:")
print(y_rf.value_counts())

In [ ]:
# 2.1: OOB sweep for n_estimators

from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import matplotlib.pyplot as plt

n_grid = [50, 100, 200, 300, 500, 800, 1000]
oob_n_results = []

for n in n_grid:
    rf = RandomForestClassifier(
        n_estimators=n,
        max_depth=None,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_n_results.append({
        "n_estimators": n,
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_n_df = pd.DataFrame(oob_n_results)
oob_n_df

In [ ]:
n_grid = [100,120,140,160,180,200]
oob_n_results = []

for n in n_grid:
    rf = RandomForestClassifier(
        n_estimators=n,
        max_depth=None,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_n_results.append({
        "n_estimators": n,
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_n_df = pd.DataFrame(oob_n_results)
oob_n_df

In [ ]:
n_grid = [100,105,110,115,120]
oob_n_results = []

for n in n_grid:
    rf = RandomForestClassifier(
        n_estimators=n,
        max_depth=None,
        min_samples_split=20,
        min_samples_leaf=10,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_n_results.append({
        "n_estimators": n,
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_n_df = pd.DataFrame(oob_n_results)
oob_n_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(oob_n_df["n_estimators"], oob_n_df["oob_accuracy"], marker='o')
plt.xlabel("n_estimators")
plt.ylabel("OOB Accuracy")
plt.title("OOB Accuracy vs Number of Trees")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 2.2: OOB sweep for max_depth

depth_grid = [3, 5, 8, 10, 15, 20, None]
oob_depth_results = []


best_n_from_oob = 115

for d in depth_grid:
    rf = RandomForestClassifier(
        n_estimators=int(best_n_from_oob),
        max_depth=d,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_depth_results.append({
        "max_depth": str(d),
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_depth_df = pd.DataFrame(oob_depth_results)
oob_depth_df

In [ ]:
depth_grid = [ 1, 2, 3, 4, 5, None]
oob_depth_results = []


best_n_from_oob = 115

for d in depth_grid:
    rf = RandomForestClassifier(
        n_estimators=int(best_n_from_oob),
        max_depth=d,
        min_samples_split=5,
        min_samples_leaf=5,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_depth_results.append({
        "max_depth": str(d),
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_depth_df = pd.DataFrame(oob_depth_results)
oob_depth_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(oob_depth_df["max_depth"], oob_depth_df["oob_accuracy"], marker='o')
plt.xlabel("max_depth")
plt.ylabel("OOB Accuracy")
plt.title("OOB Accuracy vs Tree Depth")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 2.3.1: OOB sweep for min_samples_leaf

leaf_grid = [1, 20, 40, 60, 80, 100]
oob_leaf_results = []

# best depth 

best_depth_from_oob = 4

for leaf in leaf_grid:
    rf = RandomForestClassifier(
        n_estimators=int(best_n_from_oob),
        max_depth=best_depth_from_oob,
        min_samples_split=5,
        min_samples_leaf=leaf,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_leaf_results.append({
        "min_samples_leaf": leaf,
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })

oob_leaf_df = pd.DataFrame(oob_leaf_results)
oob_leaf_df

In [ ]:
leaf_grid = [80, 82, 84, 86, 88, 90]
oob_leaf_results = []

# best depth 

best_depth_from_oob = 4

for leaf in leaf_grid:
    rf = RandomForestClassifier(
        n_estimators=int(best_n_from_oob),
        max_depth=best_depth_from_oob,
        min_samples_split=5,
        min_samples_leaf=leaf,
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=314,
        n_jobs=-1
    )
    
    rf.fit(X_rf, y_rf)
    
    oob_leaf_results.append({
        "min_samples_leaf": leaf,
        "oob_accuracy": rf.oob_score_,
        "oob_error": 1 - rf.oob_score_
    })
oob_leaf_df1 = pd.DataFrame(oob_leaf_results)
oob_leaf_df1

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(oob_leaf_df["min_samples_leaf"], oob_leaf_df["oob_accuracy"], marker='o')
plt.xlabel("min_samples_leaf")
plt.ylabel("OOB Accuracy")
plt.title("OOB Accuracy vs Minimum Leaf Size")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 2.3.2 Cross Validation for min_samples_leaf
X = x_all_selected
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_all["cancer"])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=314)

# models: Min_Leaf = 42, 82, 2
models = {
    "Random Forest (M8_42)": Pipeline([
        ("model", RandomForestClassifier(
        n_estimators=115,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=42,
        max_features='sqrt',
        random_state=314,
        n_jobs=-1
    ))]),
    "Random Forest (M8_82)": Pipeline([
        ("model", RandomForestClassifier(
        n_estimators=115,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=82,
        max_features='sqrt',
        random_state=314,
        n_jobs=-1
    ))]),
    
    "Random Forest (M8_2)": Pipeline([
        ("model", RandomForestClassifier(
        n_estimators=115,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=314,
        n_jobs=-1
    ))])
        }

# CV comparison
results = []

for name, model in models.items():
    
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    
    results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

# results
cv_results = pd.DataFrame(results).sort_values(
    by="CV Mean Accuracy",
    ascending=False
).reset_index(drop=True)

pd.set_option("display.float_format", lambda x: f"{x:.6f}")
print(cv_results)


In [ ]:
# get final hyperparameters
best_n = int(oob_n_df.loc[oob_n_df["oob_accuracy"].idxmax(), "n_estimators"])
best_depth_raw = oob_depth_df.loc[oob_depth_df["oob_accuracy"].idxmax(), "max_depth"]
best_depth = None if best_depth_raw == "None" else int(best_depth_raw)
best_leaf = 42


print("Best from OOB screening:")
print("n_estimators =", best_n)
print("max_depth =", best_depth)
print("min_samples_leaf =", best_leaf)

In [ ]:
# 3: the best model for M8

M8_rf_model = RandomForestClassifier(
    n_estimators=115,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=42,
    max_features='sqrt',
    random_state=314,
    n_jobs=-1
)


In [ ]:

# fit model
M8_rf_model.fit(x_train_selected, y_train)

# get feature importance
importances = M8_rf_model.feature_importances_

# if x_train_selected is a DataFrame
feature_names = x_train_selected.columns

# create importance table
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# sort and keep top 20
top20 = feat_imp.sort_values(by='Importance', ascending=False).head(20)

# plot
plt.figure(figsize=(10, 8))
plt.barh(top20['Feature'][::-1], top20['Importance'][::-1])
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Cross Validation for Model Selection（for 8 models）

In [ ]:
# Cross Validation for Model Selection
X = x_all_selected
#since XGBoost的y必须是[0,1,2],统一一下
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_all["cancer"])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=314)

# models
models = {
    "Logistic Regression (M1 - M4)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(penalty='l2', solver='lbfgs', max_iter=5000))
    ]),
    
    "LDA (M5)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearDiscriminantAnalysis())
    ]),
    
    "QDA (M6)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", QuadraticDiscriminantAnalysis())
    ]),
    
    "Random Forest (M8)": Pipeline([
        ("model", RandomForestClassifier(
        n_estimators=115,
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=42,
        max_features='sqrt',
        random_state=314,
        n_jobs=-1
    ))
    ]),
    
 
    "XGBoost (M7)": best_xgb_new
}

# CV comparison
results = []

for name, model in models.items():
    
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    
    results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std()
    })

# results
cv_results = pd.DataFrame(results).sort_values(
    by="CV Mean Accuracy",
    ascending=False
).reset_index(drop=True)

print(cv_results)


In [ ]:
y_train

# After CV Selection we choose RF+XGboost
### RF train/test accuracy

In [ ]:
y_train.head()

In [ ]:
# Train accuracy of RF M8
y_pred_train_M8 = M8_rf_model.predict(x_train_selected)
acc_train_M8 = accuracy_score(y_train, y_pred_train_M8 )
print("Train RF Accuracy:", acc_train_M8)

ConfusionMatrixDisplay.from_predictions(y_train,y_pred_train_M8)
plt.title("Train RFnConfusion Matrix")
plt.show()

In [ ]:
# Test accuracy of RF M8
y_pred_test_M8 = M8_rf_model.predict(x_test_selected)
acc_test_M8 = accuracy_score(y_test["cancer"], y_pred_test_M8 )
print("Test RF Accuracy:", acc_test_M8)

ConfusionMatrixDisplay.from_predictions(y_test["cancer"],y_pred_test_M8)
plt.title("Test RFnConfusion Matrix")
plt.show()

### XGboost test/train accuracy

In [ ]:
y_train.head()
#y_encoded.head()

In [ ]:
y_encoded.head()

In [ ]:
y_train_encoded = y_train - 1
y_train_encoded.head()

In [ ]:
# Train accuracy of XGB M7
y_pred_train_M7 = best_xgb_new.predict(x_train_selected)
acc_train_M7 = accuracy_score(y_train_encoded, y_pred_train_M7 )
print("Train XGB Accuracy:", acc_train_M7)

ConfusionMatrixDisplay.from_predictions(y_train_encoded,y_pred_train_M7)
plt.title("Train XGBnConfusion Matrix")
plt.show()

In [ ]:
# Test accuracy of XGB M7
y_test_encoded = y_test -1
y_pred_test_M7 = best_xgb_new.predict(x_test_selected)
acc_test_M7 = accuracy_score(y_test_encoded["cancer"], y_pred_test_M7 )
print("Test XGB Accuracy:", acc_test_M7)

ConfusionMatrixDisplay.from_predictions(y_test_encoded["cancer"],y_pred_test_M7)
plt.title("Test XGBnConfusion Matrix")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# predictions
y_pred_train_M8 = M8_rf_model.predict(x_train_selected)
y_pred_test_M8 = M8_rf_model.predict(x_test_selected)

y_pred_train_M7 = best_xgb_new.predict(x_train_selected) + 1
y_pred_test_M7 = best_xgb_new.predict(x_test_selected) + 1

# 统一 true labels 为 1,2,3
y_train_plot = y_train
y_test_plot = y_test["cancer"]

# 如果 y_train 是 DataFrame，改成：
# y_train_plot = y_train["cancer"]

# 画图：2行2列
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# RF Train
ConfusionMatrixDisplay.from_predictions(
    y_train_plot, y_pred_train_M8,
    ax=axes[0, 0],
    normalize="true",
    colorbar=False
)
axes[0, 0].set_title("RF Train")

# RF Test
ConfusionMatrixDisplay.from_predictions(
    y_test_plot, y_pred_test_M8,
    ax=axes[0, 1],
    normalize="true",
    colorbar=False
)
axes[0, 1].set_title("RF Test")

# XGB Train
ConfusionMatrixDisplay.from_predictions(
    y_train_plot, y_pred_train_M7,
    ax=axes[1, 0],
    normalize="true",
    colorbar=False
)
axes[1, 0].set_title("XGB Train")

# XGB Test
ConfusionMatrixDisplay.from_predictions(
    y_test_plot, y_pred_test_M7,
    ax=axes[1, 1],
    normalize="true",
    colorbar=False
)
axes[1, 1].set_title("XGB Test")

plt.tight_layout()
plt.show()

### find RF is the best since less overfitting compared to XGB

# find mislable 

In [ ]:


proba_cv_8 = cross_val_predict(
    M8_rf_model,
    x_all_selected,
    y_all['cancer'],
    cv=5,
    method="predict_proba")

#_______________________________________

classes_8 = M8_rf_model.classes_ # array([1, 2, 3])
pred_cv_8 = classes_8[np.argmax(proba_cv_8, axis=1)]

confidence_8 = proba_cv_8.max(axis=1)


#_______________________________________

check_8 = pd.DataFrame({
    "true_label": y_all['cancer'],
    "pred": pred_cv_8,
    "confidence_8": confidence_8 })

#_______________________________________

wrong_8 = check_8[check_8.true_label != check_8.pred]
suspect_8 = wrong_8[wrong_8["confidence_8"] > 0.85].sort_values("confidence_8", ascending=False)
suspect_8.shape


In [ ]:

y_all_encoded.head()

In [ ]:
y_all_encoded =  y_all - 1

proba_cv_7 = cross_val_predict(
    best_xgb_new,
    x_all_selected,
    y_all_encoded['cancer'],
    cv=5,
    method="predict_proba")

#_______________________________________

classes_7 = best_xgb_new.classes_ # array([1, 2, 3])
pred_cv_7 = classes_7[np.argmax(proba_cv_7, axis=1)]

confidence_7 = proba_cv_7.max(axis=1)


#_______________________________________

check_7 = pd.DataFrame({
    "true_label": y_all_encoded['cancer'],
    "pred": pred_cv_7,
    "confidence_7": confidence_7 })

#_______________________________________

wrong_7 = check_7[check_7.true_label != check_7.pred]
suspect_7 = wrong_7[wrong_7["confidence_7"] > 0.85].sort_values("confidence_7", ascending=False)
suspect_7.shape


In [ ]:
# 找 index 重合
overlap_idx = suspect_7.index.intersection(suspect_8.index)

print("M7 suspects:", suspect_7.shape[0])
print("M8 suspects:", suspect_8.shape[0])
print("Overlap:", len(overlap_idx))

In [ ]:
suspect_8

In [ ]:
wrong_8 = check_8[check_8.true_label != check_8.pred]
wrong_8

# 比较confusion matrix before drop mis vs. after drop mis using M8


In [ ]:
wrong_rest = wrong_8.drop(index=suspect_8.index)
wrong_rest.shape
x_all_selected_dropmis = x_all_selected.drop(index=suspect_8.index)
y_all_dropmis = y_all.drop(index=suspect_8.index)

In [ ]:
# accuracy of RF M8 on all data after drop mislabled
y_pred_all_M8_drop = M8_rf_model.predict(x_all_selected_dropmis)
acc_all_M8_drop = accuracy_score(y_all_dropmis["cancer"], y_pred_all_M8_drop )
print("Train RF Accuracy after drop mislabled:", acc_all_M8_drop)

ConfusionMatrixDisplay.from_predictions(y_all_dropmis["cancer"],y_pred_all_M8_drop)
plt.title("RFnConfusion Matrix after drop mis on all data")
plt.show()

#### NOTE：After removing suspected mislabeled samples, class 2 remains the most difficult class to classify, although its error rate decreases substantially. This indicates that while label noise contributed significantly to the original misclassification, there is still inherent overlap between class 2 and other cancer types, particularly class 1.

In [ ]:
# accuracy of RF M8 on all data before drop mislabled
y_pred_all_M8 = M8_rf_model.predict(x_all_selected)
acc_all_M8 = accuracy_score(y_all["cancer"], y_pred_all_M8 )
print("Train RF Accuracy after drop mislabled:", acc_all_M8)

ConfusionMatrixDisplay.from_predictions(y_all["cancer"],y_pred_all_M8)
plt.title("RFnConfusion Matrix before drop mis on all data")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

# predictions
y_pred_all_M8 = M8_rf_model.predict(x_all_selected)
y_pred_all_M8_drop = M8_rf_model.predict(x_all_selected_dropmis)

# accuracy
print("RF Accuracy BEFORE drop:", accuracy_score(y_all["cancer"], y_pred_all_M8))
print("RF Accuracy AFTER drop:", accuracy_score(y_all_dropmis["cancer"], y_pred_all_M8_drop))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#前
ConfusionMatrixDisplay.from_predictions(
    y_all["cancer"], y_pred_all_M8,
    ax=axes[0],
    normalize="true",
    colorbar=False
)
axes[0].set_title("Before Removing Mislabeled Data")

#后
ConfusionMatrixDisplay.from_predictions(
    y_all_dropmis["cancer"], y_pred_all_M8_drop,
    ax=axes[1],
    normalize="true",
    colorbar=False
)
axes[1].set_title("After Removing Mislabeled Data")

plt.tight_layout()
plt.show()

### 删除 suspected mislabeled data 后，错误显著减少
### 说明之前 confusion matrix 里的错误很大一部分来自 label noise，而不是模型能力问题
### class 2 改善最大 从 64% → 81%; class 1 也明显改善

In [ ]:

import pandas as pd

df1 = pd.read_csv("submission_m8.csv")
df2 = pd.read_csv("submission_xgb_new.csv")

print(df1.values.tolist() == df2.values.tolist())

In [ ]:
print((df1.values == df2.values).all())

In [ ]:
df1.head

In [ ]:
df2.head

In [ ]:


# 统一列名（关键！）
df1 = df1.rename(columns={"cancer": "label"})
# 如果df2不是label也可以统一
# df2 = df2.rename(columns={"xxx": "label"})

# 按 id merge
merged = df1.merge(df2, on="id", suffixes=("_m8", "_xgb"))

# 比较是否完全一致
print((merged["label_m8"] == merged["label_xgb"]).all())

In [ ]:
diff = merged[merged["label_m8"] != merged["label_xgb"]]
print(diff)